# Sending email with `htp.mail`

The `htp.mail` module exposes two small helpers on top of `smtplib` and `email.mime`: `send_mail` for a plain-text message, and `send_mail_with_attachment` for a message with a single file attached. Both take the SMTP server hostname as an explicit argument and use `with smtplib.SMTP(...)` so the connection is closed cleanly on failure.

The code cells below are illustrative. They are not executed when the book builds, because they need a reachable SMTP server.

## Importing

Both functions are re-exported from the top level of the package.

In [ ]:
from htp import send_mail, send_mail_with_attachment

## A plain-text message to one recipient

Pass the recipient address as a string. `send_mail` opens an SMTP connection to `mailserver`, hands the message off, and closes the connection.

In [ ]:
send_mail(
    toaddrs="alice@example.com",
    subject="nightly run finished",
    message="All 42 jobs completed successfully.",
    fromaddr="runner@example.com",
    mailserver="smtp.example.com",
)

## Several recipients at once

Pass an iterable of addresses as `toaddrs`. A single message is sent, and every address is listed in the `To:` header, so each recipient can see the others.

In [ ]:
send_mail(
    toaddrs=["alice@example.com", "bob@example.com"],
    subject="nightly run finished",
    message="All 42 jobs completed successfully.",
    fromaddr="runner@example.com",
    mailserver="smtp.example.com",
)

## Attaching a file

`send_mail_with_attachment` sends one message per recipient so that addresses are not exposed to each other in the headers. The `attachment` argument is a path to a file on the local filesystem; the file is read, base64-encoded, and attached as `application/octet-stream` with its basename as the displayed filename.

In [ ]:
send_mail_with_attachment(
    fromaddr="runner@example.com",
    toaddrs=["alice@example.com", "bob@example.com"],
    subject="nightly run log",
    message="Log from the 2026-04-13 run is attached.",
    attachment="logs/2026-04-13.log",
    mailserver="smtp.example.com",
)

## Notes and caveats

Authentication and TLS are not wired in. `send_mail` and `send_mail_with_attachment` use a bare `smtplib.SMTP(mailserver)` connection, which assumes the SMTP server accepts unauthenticated delivery from the machine the script runs on (for example an internal relay). If your provider requires login or STARTTLS, call `smtplib` directly and adapt the MIME-construction logic from these helpers.

The body is always plain text. For HTML mail, or for multiple attachments, build a `MIMEMultipart` by hand.